In [ ]:
import subprocess, os, sys, time, json, shutil
def sh(c, t=3600):
    print(f"$ {c[:160]}", flush=True)
    r = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=t)
    out = (r.stdout or "") + (r.stderr or "")
    print(out[-4000:], flush=True)
    return r.returncode, out

t0 = time.time()
os.makedirs("/root/.hermes/logs", exist_ok=True)
env = dict(os.environ, HERMES_HOME="/root/.hermes", NO_COLOR="1", CI="1")
cmd = ("curl -fsSL https://hermes-agent.nousresearch.com/install.sh | bash -s -- "
       "--non-interactive --verbose")
r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=4200, env=env)
tail = ((r.stdout or "") + (r.stderr or ""))
print(tail[-12000:], flush=True)
print("INSTALL_RC", r.returncode, "SECONDS", round(time.time()-t0,1), flush=True)

In [ ]:
import subprocess, os, json
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=600)
    return ((r.stdout or "") + (r.stderr or "")).strip()
print("hermes bin:", sh("command -v hermes; ls -la /root/.local/bin/hermes 2>&1"))
print("version:", sh("/root/.local/bin/hermes --version 2>&1 | tail -2"))
print("tree:", sh("du -sh /root/.hermes 2>/dev/null; du -sh /root/.hermes/* 2>/dev/null | sort -h"))
print("which zstd:", sh("command -v zstd; tar --version | head -1"))
print("venv:", sh("ls -la /root/.hermes/hermes-agent 2>&1 | head -20"))
print("pm tools:", sh("ls -la /root/.hermes/pm 2>&1 | head; du -sh /root/.hermes/pm/* 2>/dev/null | sort -h"))
print("total root fs:", sh("df -h /kaggle/working | tail -1"))

In [ ]:
import subprocess, os, json, time
def sh(c, t=3600):
    r = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=t)
    return ((r.stdout or "") + (r.stderr or "")).strip()

t0=time.time()
zstd = sh("command -v zstd")
out = "/kaggle/working/hermes-agent-full.pkg"
if zstd:
    cmd = f"tar -C / -cf - --numeric-owner root/.hermes root/.local 2>/dev/null | zstd -19 -T0 -q -o {out}"
else:
    cmd = f"tar -C / -czf {out} --numeric-owner root/.hermes root/.local"
print(sh(f"ls -la /root/.local/bin | head -30"))
rc = sh(cmd + "; echo RC=$?")
print(rc[-2000:])
print("size:", sh(f"du -h {out}").split()[0] if os.path.exists(out) else "MISSING", round(time.time()-t0,1),"s")
# manifest
man = {"built_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
       "python": sh("python3 -V"), "os": sh("cat /etc/os-release | head -2"),
       "hermes_version": sh("/root/.local/bin/hermes --version 2>&1 | head -1"),
       "du": sh("du -so /root/.hermes /root/.local 2>/dev/null"),
       "archive_bytes": os.path.getsize(out) if os.path.exists(out) else 0}
open("/kaggle/working/manifest.json","w").write(json.dumps(man, indent=1))
print(json.dumps(man, indent=1))